In [1]:
import pandas as pd
import numpy as np
from google import genai
from google.genai import types
import json
import base64
from typing import Dict, List
import os
import anthropic
import openai

In [ ]:
def create_image(filepath: str) -> str:
    with open(filepath, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [3]:
def create_word_data(word: str, image_paths: List[str], options: Dict[str, str]) -> Dict:
   
    return {
        "word": word,
        "images": image_paths,
        "options": options
    }

In [4]:
def get_prompt(prompt_type: str, word: str, options: Dict[str, str]) -> str:
    
    options_text = "\n".join([f"{k}) {v}" for k, v in options.items()])
    
    if prompt_type == "minimal":
        return f"""You will see 5 images where a native speaker used the word "{word}".

What does "{word}" mean?

{options_text}

Answer with ONLY the letter (a, b, c, d, or e)."""

    elif prompt_type == "elaborate":
        return f"""You are a linguist documenting an unknown language. You have no prior knowledge of how these speakers categorize reality.

A native speaker used the word "{word}" when pointing at objects in these 5 situations.

Based ONLY on observable patterns in the images - without assuming any framework for categorizing reality - what does "{word}" most likely mean?

{options_text}

Answer with ONLY the letter (a, b, c, d, or e)."""

    else:
        raise ValueError(f"Unknown prompt_type: {prompt_type}. Use 'minimal' or 'elaborate'")

In [ ]:
google_api_key = os.getenv("GOOGLE_API_KEY")


def call_gemini(prompt: str, images: List[str], api_key: str, 
                model: str = "gemini-2.5-flash") -> str:
    client = genai.Client(api_key=api_key)
    
    contents = [prompt]
    for img_data in images:
        contents.append({
            "inline_data": {"mime_type": "image/jpeg", "data": img_data}
        })
    
    response = client.models.generate_content(
        model=model,
        contents=contents
    )
    
    return response.text.strip()

In [ ]:
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

def call_claude(prompt: str, images: List[str], api_key: str,
                model: str = "claude-sonnet-4-5-20250929") -> str:
    client = anthropic.Anthropic(api_key=api_key)
    
    content = []
    for img_data in images:
        content.append({
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/jpeg",
                "data": img_data
            }
        })
    content.append({"type": "text", "text": prompt})
    
    response = client.messages.create(
        model=model,
        max_tokens=300,
        messages=[{"role": "user", "content": content}]
    )
    
    return response.content[0].text.strip()

In [ ]:
openai_api_key = os.getenv("OPENAI_API_KEY")

def call_openai(prompt: str, images: List[str], api_key: str,
                model: str = "gpt-4o") -> str:
    client = openai.OpenAI(api_key=api_key)
    
    content = [{"type": "text", "text": prompt}]
    for img_data in images:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{img_data}"}
        })
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": content}],
        max_tokens=300
    )
    
    return response.choices[0].message.content.strip()

In [8]:
def run_experiment(
    word_data: Dict,
    model_name: str,
    api_key: str,
    prompt_type: str,
    n_iterations: int = 10,
) -> Dict:
    
    word = word_data["word"]
    image_paths = word_data["images"]
    options = word_data["options"]
    
    print(f"Encoding {len(image_paths)} images for '{word}'...")
    encoded_images = [create_image(path) for path in image_paths]
    
    prompt = get_prompt(prompt_type, word, options)
    
    api_functions = {
        "gemini": call_gemini,
        "claude": call_claude,
        "gpt4": call_openai
    }
    
    if model_name not in api_functions:
        raise ValueError(f"Unknown model: {model_name}. Use 'gemini', 'claude', or 'gpt4'")
    
    api_call = api_functions[model_name]
    
    results = []
    print(f"\nRunning {n_iterations} iterations for '{word}' with {model_name} ({prompt_type} prompt)...")
    
    for i in range(n_iterations):
        try:
            response = api_call(prompt, encoded_images, api_key)
            results.append(response)
            print(f"  Trial {i+1}/{n_iterations}: {response}")
            
        except Exception as e:
            print(f"  Trial {i+1}/{n_iterations}: ERROR - {e}")
            results.append(f"ERROR: {e}")
    
    return {
        "word": word,
        "model": model_name,
        "prompt_type": prompt_type,
        "n_iterations": n_iterations,
        "responses": results}

In [9]:
julpa_data = create_word_data(word ='julpa', image_paths =  ['main_quine_file/quine_images/book_images/single_page_from_a_book.jpg', 
                                              'main_quine_file/quine_images/book_images/person_reading_a_book.jpg',
                                             'main_quine_file/quine_images/book_images/open_book.jpg',
                                             'main_quine_file/quine_images/book_images/coloring_book.jpg',
                                             'main_quine_file/quine_images/book_images/closed_hard_cover_book.jpg'], options = {'a': 'book',
                                                                                                                               'b': 'reading',
                                                                                                                               'c': 'paper',
                                                                                                                               'd': 'literature',
                                                                                                                               'e': 'page'})

run_experiment(word_data = julpa_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'julpa',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [10]:
run_experiment(word_data = julpa_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with gemini (elaborate prompt)...
  Trial 1/10: a)
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a) book
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'julpa',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a)', 'a', 'a', 'a', 'a', 'a', 'a) book', 'a', 'a', 'a']}

In [49]:
run_experiment(word_data = julpa_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with claude (minimal prompt)...
  Trial 1/10: Looking at the 5 images, I can see:
1. An open book showing a text page
2. A person reading a book
3. An open book on a table
4. A coloring book with drawings
5. A closed book on a dark surface

The common element across all images is **book** - they all show books in different states (open, closed, being read, etc.).

**a**
  Trial 2/10: Looking at the 5 images, I can see:
1. An open book showing a page with text
2. A person reading a book
3. An open book with pages visible
4. A coloring book or activity book open on a table
5. A closed book on a table

The word "julpa" appears in all these contexts. While books and reading are prominent, the most consistent element across all images is the physical book itself - whether open or closed, being read or sitting on a surface.

**a**
  Trial 3/10: Looking at the 5 images, I can see:
1. An open book showing a page with text
2. A

{'word': 'julpa',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at the 5 images, I can see:\n1. An open book showing a text page\n2. A person reading a book\n3. An open book on a table\n4. A coloring book with drawings\n5. A closed book on a dark surface\n\nThe common element across all images is **book** - they all show books in different states (open, closed, being read, etc.).\n\n**a**',
  'Looking at the 5 images, I can see:\n1. An open book showing a page with text\n2. A person reading a book\n3. An open book with pages visible\n4. A coloring book or activity book open on a table\n5. A closed book on a table\n\nThe word "julpa" appears in all these contexts. While books and reading are prominent, the most consistent element across all images is the physical book itself - whether open or closed, being read or sitting on a surface.\n\n**a**',
  'Looking at the 5 images, I can see:\n1. An open book showing a page with text\n2. A person re

In [50]:
run_experiment(word_data = julpa_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with claude (elaborate prompt)...
  Trial 1/10: I need to analyze what common observable feature appears in all 5 images where "julpa" was used.

Image 1: An open book showing printed pages
Image 2: A person holding and reading an open book
Image 3: An open book lying on a surface
Image 4: An open coloring book with drawings and colored pencils
Image 5: A closed book on a surface

Observable pattern: In images 1, 2, 3, and 4, there are books in an OPEN state showing pages. Image 5 shows a CLOSED book. However, all images contain a book object.

Let me reconsider: The most consistent physical object present in ALL 5 situations is a book (bound collection of pages). 

- "Reading" (b) - only clearly happening in image 2
- "Paper" (c) - visible in images 1-4 but not the focus in image 5
- "Literature" (d) - too abstract to observe
- "Page" (e) - visible in images 1-4, but image 5 shows closed book
- "Book" (a) - present in

{'word': 'julpa',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['I need to analyze what common observable feature appears in all 5 images where "julpa" was used.\n\nImage 1: An open book showing printed pages\nImage 2: A person holding and reading an open book\nImage 3: An open book lying on a surface\nImage 4: An open coloring book with drawings and colored pencils\nImage 5: A closed book on a surface\n\nObservable pattern: In images 1, 2, 3, and 4, there are books in an OPEN state showing pages. Image 5 shows a CLOSED book. However, all images contain a book object.\n\nLet me reconsider: The most consistent physical object present in ALL 5 situations is a book (bound collection of pages). \n\n- "Reading" (b) - only clearly happening in image 2\n- "Paper" (c) - visible in images 1-4 but not the focus in image 5\n- "Literature" (d) - too abstract to observe\n- "Page" (e) - visible in images 1-4, but image 5 shows closed book\n- "Book" (a) - presen

In [9]:
run_experiment(word_data = julpa_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with gpt4 (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: c) paper
  Trial 3/10: a
  Trial 4/10: a) book
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: c) paper
  Trial 10/10: a


{'word': 'julpa',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a',
  'c) paper',
  'a',
  'a) book',
  'a',
  'a',
  'a',
  'a',
  'c) paper',
  'a']}

In [10]:
run_experiment(word_data = julpa_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'julpa'...

Running 10 iterations for 'julpa' with gpt4 (elaborate prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a) book
  Trial 10/10: a


{'word': 'julpa',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a) book', 'a']}

In [10]:
blirox_data = create_word_data(word ='blirox', image_paths =  ['main_quine_file/quine_images/door_images/bank_vault_door.jpg', 
                                              'main_quine_file/quine_images/door_images/body_of_a_door.jpg',
                                             'main_quine_file/quine_images/door_images/glass_door.jpg',
                                             'main_quine_file/quine_images/door_images/doorknob.jpg',
                                             'main_quine_file/quine_images/door_images/white_door.jpg'], options = {'a': 'door',
                                                                                                                    'b': 'opening',
                                                                                                                    'c': 'rectangle',
                                                                                                                    'd': 'portal',
                                                                                                                    'e': 'doorknob'})

run_experiment(word_data = blirox_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'blirox',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [12]:
run_experiment(word_data = blirox_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with gemini (elaborate prompt)...
  Trial 1/10: The task is to identify the most likely meaning of "blirox" based *only* on observable patterns in the 5 images, without assuming any prior frameworks.

Let's analyze each image and how the options fit:

1.  **Image 1 (Vault door):** This is a large, metallic, complex door.
    *   a) door: Fits perfectly.
    *   b) opening: Does not fit; it's a closed barrier.
    *   c) rectangle: The overall door is rectangular, but there are many circular components.
    *   d) portal: Too abstract; it's a physical door.
    *   e) doorknob: Does not fit; it has wheels/levers, not a conventional doorknob.

2.  **Image 2 (White interior door):** This is a standard closed door.
    *   a) door: Fits perfectly.
    *   b) opening: Does not fit; it's a closed barrier.
    *   c) rectangle: The door is rectangular.
    *   d) portal: Too abstract.
    *   e) doorknob: No doorknob is vis

{'word': 'blirox',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['The task is to identify the most likely meaning of "blirox" based *only* on observable patterns in the 5 images, without assuming any prior frameworks.\n\nLet\'s analyze each image and how the options fit:\n\n1.  **Image 1 (Vault door):** This is a large, metallic, complex door.\n    *   a) door: Fits perfectly.\n    *   b) opening: Does not fit; it\'s a closed barrier.\n    *   c) rectangle: The overall door is rectangular, but there are many circular components.\n    *   d) portal: Too abstract; it\'s a physical door.\n    *   e) doorknob: Does not fit; it has wheels/levers, not a conventional doorknob.\n\n2.  **Image 2 (White interior door):** This is a standard closed door.\n    *   a) door: Fits perfectly.\n    *   b) opening: Does not fit; it\'s a closed barrier.\n    *   c) rectangle: The door is rectangular.\n    *   d) portal: Too abstract.\n    *   e) doorknob: No doorknob

In [13]:
run_experiment(word_data = blirox_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with claude (minimal prompt)...
  Trial 1/10: Looking at all 5 images, I can see they all feature doors prominently - a vault door, interior doors, a glass door, and doors with various hardware. The word "blirox" appears to be used by the native speaker to refer to these structures.

While doorknobs appear in some images, they are not present in all of them (like the vault door and glass door). The most consistent element across all images is the door itself as a complete structure.

**a**
  Trial 2/10: Looking at the 5 images, I can see:
1. A vault door (heavy bank-style door)
2. A white interior door with panels
3. A glass office door
4. A doorknob with keyhole on a white door
5. A white interior door in a hallway

The common element across all images is "door" - they all show different types of doors. While image 4 focuses on a doorknob, it's still part of a door. "Opening," "rectangle," and "portal" are too gener

{'word': 'blirox',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at all 5 images, I can see they all feature doors prominently - a vault door, interior doors, a glass door, and doors with various hardware. The word "blirox" appears to be used by the native speaker to refer to these structures.\n\nWhile doorknobs appear in some images, they are not present in all of them (like the vault door and glass door). The most consistent element across all images is the door itself as a complete structure.\n\n**a**',
  'Looking at the 5 images, I can see:\n1. A vault door (heavy bank-style door)\n2. A white interior door with panels\n3. A glass office door\n4. A doorknob with keyhole on a white door\n5. A white interior door in a hallway\n\nThe common element across all images is "door" - they all show different types of doors. While image 4 focuses on a doorknob, it\'s still part of a door. "Opening," "rectangle," and "portal" are too general or abst

In [14]:
blirox_data = create_word_data(word ='blirox', image_paths =  ['main_quine_file/quine_images/door_images/bank_vault_door.jpg', 
                                              'main_quine_file/quine_images/door_images/body_of_a_door.jpg',
                                             'main_quine_file/quine_images/door_images/glass_door.jpg',
                                             'main_quine_file/quine_images/door_images/doorknob.jpg',
                                             'main_quine_file/quine_images/door_images/white_door.jpg'], options = {'a': 'door',
                                                                                                                    'b': 'opening',
                                                                                                                    'c': 'rectangle',
                                                                                                                    'd': 'portal',
                                                                                                                    'e': 'doorknob'})

run_experiment(word_data = blirox_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with claude (elaborate prompt)...
  Trial 1/10: I need to analyze what observable pattern connects all 5 images where "blirox" was used.

Image 1: A heavy vault door with complex locking mechanisms
Image 2: A simple white residential door with panels
Image 3: A modern glass door (appears to be sliding/office style)
Image 4: A close-up of a doorknob with keyhole on a white door
Image 5: A white paneled door in a hallway/stairway setting

Observable patterns:
- All images show doors (specifically)
- All have some kind of opening function
- All contain rectangles, but so do many other objects
- All could be considered portals in an abstract sense
- Only image 4 prominently features a doorknob

The most consistent observable element across all 5 images is the presence of a door as the primary object. While doorknobs, rectangles, openings, and portals are present or implied, "door" is the most specific and consistent patt

{'word': 'blirox',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['I need to analyze what observable pattern connects all 5 images where "blirox" was used.\n\nImage 1: A heavy vault door with complex locking mechanisms\nImage 2: A simple white residential door with panels\nImage 3: A modern glass door (appears to be sliding/office style)\nImage 4: A close-up of a doorknob with keyhole on a white door\nImage 5: A white paneled door in a hallway/stairway setting\n\nObservable patterns:\n- All images show doors (specifically)\n- All have some kind of opening function\n- All contain rectangles, but so do many other objects\n- All could be considered portals in an abstract sense\n- Only image 4 prominently features a doorknob\n\nThe most consistent observable element across all 5 images is the presence of a door as the primary object. While doorknobs, rectangles, openings, and portals are present or implied, "door" is the most specific and consistent pa

In [15]:
run_experiment(word_data = blirox_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with gpt4 (minimal prompt)...
  Trial 1/10: a) door
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'blirox',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a) door', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [16]:
run_experiment(word_data = blirox_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'blirox'...

Running 10 iterations for 'blirox' with gpt4 (elaborate prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: c
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'blirox',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a', 'a', 'c', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [17]:
varnon_data = create_word_data(word ='varnon', image_paths =  ['main_quine_file/quine_images/chair_images/car_seat.jpg', 
                                              'main_quine_file/quine_images/chair_images/chair.jpg',
                                             'main_quine_file/quine_images/chair_images/throne.jpg',
                                             'main_quine_file/quine_images/chair_images/zoomed_in_chair_handle.jpg',
                                             'main_quine_file/quine_images/chair_images/zoomed_in_chair_seat.jpg'], options = {'a': 'chair',
                                                                                                                    'b': 'sitting',
                                                                                                                    'c': 'comfort',
                                                                                                                    'd': 'furniture',
                                                                                                                    'e': 'backrest'})

run_experiment(word_data = varnon_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'varnon',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [18]:
run_experiment(word_data = varnon_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with gemini (elaborate prompt)...
  Trial 1/10: a)
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a)
  Trial 5/10: a)
  Trial 6/10: a
  Trial 7/10: a)
  Trial 8/10: a)
  Trial 9/10: a)
  Trial 10/10: a


{'word': 'varnon',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a)', 'a', 'a', 'a)', 'a)', 'a', 'a)', 'a)', 'a)', 'a']}

In [19]:
run_experiment(word_data = varnon_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with claude (minimal prompt)...
  Trial 1/10: Looking at these 5 images, I can see various types of seating items:
1. A car seat
2. A wooden chair
3. An ornate throne
4. A close-up of an upholstered chair arm/detail
5. A cushioned seat detail

The common element across all images is "chair" - they all show different types of chairs or seats. The word "varnon" appears to refer to the object itself rather than concepts like sitting, comfort, or just a part like the backrest.

**a**
  Trial 2/10: Looking at these 5 images, I can see:
1. A car seat
2. A wooden chair
3. An ornate throne
4. A close-up of a chair arm/seat
5. A close-up of a chair cushion

All of these images show different types of seats or seating furniture. The common element across all images is "chair" - they are all variations of chairs or seats where people sit.

**a**
  Trial 3/10: I need to analyze these 5 images to find the common element that woul

{'word': 'varnon',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at these 5 images, I can see various types of seating items:\n1. A car seat\n2. A wooden chair\n3. An ornate throne\n4. A close-up of an upholstered chair arm/detail\n5. A cushioned seat detail\n\nThe common element across all images is "chair" - they all show different types of chairs or seats. The word "varnon" appears to refer to the object itself rather than concepts like sitting, comfort, or just a part like the backrest.\n\n**a**',
  'Looking at these 5 images, I can see:\n1. A car seat\n2. A wooden chair\n3. An ornate throne\n4. A close-up of a chair arm/seat\n5. A close-up of a chair cushion\n\nAll of these images show different types of seats or seating furniture. The common element across all images is "chair" - they are all variations of chairs or seats where people sit.\n\n**a**',
  'I need to analyze these 5 images to find the common element that would be called "

In [20]:
run_experiment(word_data = varnon_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with claude (elaborate prompt)...
  Trial 1/10: I need to analyze what is consistently present across all 5 images where "varnon" was used.

Image 1: A car seat with a headrest and backrest
Image 2: A wooden chair with a backrest
Image 3: An ornate throne with a tall backrest
Image 4: A chair showing the armrest and backrest detail
Image 5: A chair showing the seat cushion detail

Observable patterns:
- All images show objects designed for sitting
- All have a seat surface
- All have a backrest (visible or implied structure)
- All are individual seating objects
- Not all show the act of sitting (no people)
- "Comfort" is not directly observable
- "Furniture" would be too broad
- "Backrest" is present in all but is just a component

The most consistent observable feature is that all objects are chairs (or chair-like seating objects for one person with a backrest and seat).

**a**
  Trial 2/10: I need to analyze what i

{'word': 'varnon',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['I need to analyze what is consistently present across all 5 images where "varnon" was used.\n\nImage 1: A car seat with a headrest and backrest\nImage 2: A wooden chair with a backrest\nImage 3: An ornate throne with a tall backrest\nImage 4: A chair showing the armrest and backrest detail\nImage 5: A chair showing the seat cushion detail\n\nObservable patterns:\n- All images show objects designed for sitting\n- All have a seat surface\n- All have a backrest (visible or implied structure)\n- All are individual seating objects\n- Not all show the act of sitting (no people)\n- "Comfort" is not directly observable\n- "Furniture" would be too broad\n- "Backrest" is present in all but is just a component\n\nThe most consistent observable feature is that all objects are chairs (or chair-like seating objects for one person with a backrest and seat).\n\n**a**',
  'I need to analyze what is 

In [21]:
run_experiment(word_data = varnon_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with gpt4 (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a) chair
  Trial 3/10: a) chair
  Trial 4/10: a) chair
  Trial 5/10: a) chair
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a) chair
  Trial 9/10: a
  Trial 10/10: a


{'word': 'varnon',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a',
  'a) chair',
  'a) chair',
  'a) chair',
  'a) chair',
  'a',
  'a',
  'a) chair',
  'a',
  'a']}

In [22]:
run_experiment(word_data = varnon_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'varnon'...

Running 10 iterations for 'varnon' with gpt4 (elaborate prompt)...
  Trial 1/10: e) backrest
  Trial 2/10: a
  Trial 3/10: e) backrest
  Trial 4/10: a
  Trial 5/10: e) backrest
  Trial 6/10: c
  Trial 7/10: a
  Trial 8/10: e
  Trial 9/10: a
  Trial 10/10: e) backrest


{'word': 'varnon',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['e) backrest',
  'a',
  'e) backrest',
  'a',
  'e) backrest',
  'c',
  'a',
  'e',
  'a',
  'e) backrest']}

In [23]:
flim_data = create_word_data(word ='flim', image_paths =  ['main_quine_file/quine_images/mug_images/mug_bottom.jpg', 
                                              'main_quine_file/quine_images/mug_images/mug_handle.jpg',
                                             'main_quine_file/quine_images/mug_images/mug.jpg',
                                             'main_quine_file/quine_images/mug_images/spiderman_mug.jpg',
                                             'main_quine_file/quine_images/mug_images/toy_mug.jpg'], options = {'a': 'mug',
                                                                                                                    'b': 'drinking',
                                                                                                                    'c': 'tube',
                                                                                                                    'd': 'container',
                                                                                                                    'e': 'handle'})

run_experiment(word_data = flim_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a)
  Trial 4/10: a
  Trial 5/10: e) handle
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'flim',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a)', 'a', 'e) handle', 'a', 'a', 'a', 'a', 'a']}

In [24]:
run_experiment(word_data = flim_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with gemini (elaborate prompt)...
  Trial 1/10: a)
  Trial 2/10: e)
  Trial 3/10: e
  Trial 4/10: e)
  Trial 5/10: e) handle
  Trial 6/10: e
  Trial 7/10: e) handle
  Trial 8/10: a)
  Trial 9/10: e
  Trial 10/10: e) handle


{'word': 'flim',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a)',
  'e)',
  'e',
  'e)',
  'e) handle',
  'e',
  'e) handle',
  'a)',
  'e',
  'e) handle']}

In [25]:
run_experiment(word_data = flim_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with claude (minimal prompt)...
  Trial 1/10: Looking at all 5 images, I can see they all feature mugs/cups prominently. The images show:
1. Bottom view of a dark brown mug
2. Close-up of a brown mug's handle
3. A two-tone mug with coffee on a wooden surface
4. A Spider-Man character mug
5. A red brick-patterned mug

The common element across all images is the mug itself.

**a**
  Trial 2/10: Looking at all 5 images, they all show coffee mugs - ceramic vessels with handles used for drinking beverages.

The answer is **a**
  Trial 3/10: Looking at all 5 images, they each show a different mug - a brown ceramic mug from the bottom, a brown mug's handle, a two-toned mug with coffee, a Spider-Man character mug, and a red brick-patterned mug.

The common element across all images is that they all feature mugs.

**a**
  Trial 4/10: Looking at the 5 images, they all show different types of mugs - ceramic mugs with handles used f

{'word': 'flim',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ["Looking at all 5 images, I can see they all feature mugs/cups prominently. The images show:\n1. Bottom view of a dark brown mug\n2. Close-up of a brown mug's handle\n3. A two-tone mug with coffee on a wooden surface\n4. A Spider-Man character mug\n5. A red brick-patterned mug\n\nThe common element across all images is the mug itself.\n\n**a**",
  'Looking at all 5 images, they all show coffee mugs - ceramic vessels with handles used for drinking beverages.\n\nThe answer is **a**',
  "Looking at all 5 images, they each show a different mug - a brown ceramic mug from the bottom, a brown mug's handle, a two-toned mug with coffee, a Spider-Man character mug, and a red brick-patterned mug.\n\nThe common element across all images is that they all feature mugs.\n\n**a**",
  'Looking at the 5 images, they all show different types of mugs - ceramic mugs with handles used for drinking beverages l

In [26]:
run_experiment(word_data = flim_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with claude (elaborate prompt)...
  Trial 1/10: I need to identify what observable feature is common across all 5 images.

Image 1: Shows a dark brown mug from bottom view, showing its base and handle
Image 2: Shows a brown mug's handle in close-up
Image 3: Shows a brown/beige mug with coffee on a wooden surface
Image 4: Shows a Spider-Man novelty mug
Image 5: Shows a red brick-pattern mug

Observable patterns:
- All images show mugs (objects with cylindrical bodies and handles)
- All have handles visible or implied
- All are containers
- Not all show drinking (no person drinking)
- Handles are curved/tubular appendages

The most specific observable pattern that appears in ALL images is the presence of a **handle** - a curved, tubular protrusion attached to the side of each object.

While "mug," "container," and "handle" are all present, "handle" is the most distinctive common feature across all examples, especially give

{'word': 'flim',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['I need to identify what observable feature is common across all 5 images.\n\nImage 1: Shows a dark brown mug from bottom view, showing its base and handle\nImage 2: Shows a brown mug\'s handle in close-up\nImage 3: Shows a brown/beige mug with coffee on a wooden surface\nImage 4: Shows a Spider-Man novelty mug\nImage 5: Shows a red brick-pattern mug\n\nObservable patterns:\n- All images show mugs (objects with cylindrical bodies and handles)\n- All have handles visible or implied\n- All are containers\n- Not all show drinking (no person drinking)\n- Handles are curved/tubular appendages\n\nThe most specific observable pattern that appears in ALL images is the presence of a **handle** - a curved, tubular protrusion attached to the side of each object.\n\nWhile "mug," "container," and "handle" are all present, "handle" is the most distinctive common feature across all examples, especial

In [27]:
run_experiment(word_data = flim_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with gpt4 (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'flim',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [28]:
run_experiment(word_data = flim_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'flim'...

Running 10 iterations for 'flim' with gpt4 (elaborate prompt)...
  Trial 1/10: a
  Trial 2/10: a) mug
  Trial 3/10: a) mug
  Trial 4/10: a
  Trial 5/10: d) container
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a) mug
  Trial 10/10: a


{'word': 'flim',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a',
  'a) mug',
  'a) mug',
  'a',
  'd) container',
  'a',
  'a',
  'a',
  'a) mug',
  'a']}

In [29]:
tafno_data = create_word_data(word ='tafno', image_paths =  ['main_quine_file/quine_images/tree_images/toy_tree.jpg', 
                                              'main_quine_file/quine_images/tree_images/artist_tree.jpg',
                                             'main_quine_file/quine_images/tree_images/tree_branch.jpg',
                                             'main_quine_file/quine_images/tree_images/tree_root.jpg',
                                             'main_quine_file/quine_images/tree_images/tree.jpg'], options = {'a': 'tree',
                                                                                                                    'b': 'building material',
                                                                                                                    'c': 'wood',
                                                                                                                    'd': 'plant',
                                                                                                                    'e': 'bark'})

run_experiment(word_data = tafno_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'tafno',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [30]:
run_experiment(word_data = tafno_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with gemini (elaborate prompt)...
  Trial 1/10: a) tree
  Trial 2/10: Based ONLY on observable patterns:

1.  **Image 1:** A model of a tree made of plastic. The speaker points at the entire model.
2.  **Image 2:** A decorative item shaped like a tree, made from tree branches and leaves, mounted on a wooden board. The speaker points at the tree-shaped arrangement.
3.  **Image 3:** Branches and leaves from a tree. The speaker points at these parts.
4.  **Image 4:** The roots and lower trunk of a tree. The speaker points at these parts.
5.  **Image 5:** A complete tree in a field. The speaker points at the entire tree.

Let's evaluate the options:

*   **a) tree:** This fits all instances. It covers actual trees (Image 3, 4, 5), a decorative item shaped like a tree (Image 2), and a model of a tree (Image 1). The concept of 'tree' (including representations or parts of it) is consistently present.
*   **b) building materi

{'word': 'tafno',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a) tree',
  'Based ONLY on observable patterns:\n\n1.  **Image 1:** A model of a tree made of plastic. The speaker points at the entire model.\n2.  **Image 2:** A decorative item shaped like a tree, made from tree branches and leaves, mounted on a wooden board. The speaker points at the tree-shaped arrangement.\n3.  **Image 3:** Branches and leaves from a tree. The speaker points at these parts.\n4.  **Image 4:** The roots and lower trunk of a tree. The speaker points at these parts.\n5.  **Image 5:** A complete tree in a field. The speaker points at the entire tree.\n\nLet\'s evaluate the options:\n\n*   **a) tree:** This fits all instances. It covers actual trees (Image 3, 4, 5), a decorative item shaped like a tree (Image 2), and a model of a tree (Image 1). The concept of \'tree\' (including representations or parts of it) is consistently present.\n*   **b) building material:** O

In [31]:
run_experiment(word_data = tafno_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with claude (minimal prompt)...
  Trial 1/10: Looking at the 5 images:
1. A LEGO tree made of plastic blocks
2. A cardboard art piece with dried plants/twigs
3. A real tree with leaves (some turning colors)
4. Exposed tree roots
5. A mature tree in a field

The common element across all images is "tree" - whether artificial (LEGO), incorporated into art, or real trees in various forms (full tree, roots, solitary tree).

**a**
  Trial 2/10: a
  Trial 3/10: Looking at the 5 images:
1. A LEGO tree made of plastic blocks
2. A cardboard art piece with dried branches/twigs
3. A real tree with leaves in nature
4. Tree roots on the ground
5. A solitary tree in a field

The common element across all images is "tree" - whether artificial (LEGO), as part of art (cardboard with branches), or real trees in various contexts.

**a**
  Trial 4/10: Looking at the 5 images:

1. A LEGO tree made of plastic blocks
2. A craft/art piece wit

{'word': 'tafno',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at the 5 images:\n1. A LEGO tree made of plastic blocks\n2. A cardboard art piece with dried plants/twigs\n3. A real tree with leaves (some turning colors)\n4. Exposed tree roots\n5. A mature tree in a field\n\nThe common element across all images is "tree" - whether artificial (LEGO), incorporated into art, or real trees in various forms (full tree, roots, solitary tree).\n\n**a**',
  'a',
  'Looking at the 5 images:\n1. A LEGO tree made of plastic blocks\n2. A cardboard art piece with dried branches/twigs\n3. A real tree with leaves in nature\n4. Tree roots on the ground\n5. A solitary tree in a field\n\nThe common element across all images is "tree" - whether artificial (LEGO), as part of art (cardboard with branches), or real trees in various contexts.\n\n**a**',
  'Looking at the 5 images:\n\n1. A LEGO tree made of plastic blocks\n2. A craft/art piece with dried branches/t

In [32]:
run_experiment(word_data = tafno_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with claude (elaborate prompt)...
  Trial 1/10: Looking at the 5 images, I need to identify what observable feature is common to all objects labeled "tafno":

Image 1: LEGO tree model - has a trunk structure
Image 2: Artistic piece with branches/twigs - has a trunk-like structure
Image 3: Real tree with branches and leaves - has a trunk
Image 4: Tree roots spreading on ground - part of a tree's trunk system
Image 5: Solitary tree in a field - has a prominent trunk

The most consistent observable pattern across all images is the presence of a **tree** or tree-like structure with a trunk. 

- Not "building material" (b) - images 3 and 5 show living trees in nature
- Not "wood" (c) - image 3 shows a living tree with green leaves
- Not "plant" (d) - too broad; would include grass, flowers, etc. not shown
- Not "bark" (e) - not all images focus on bark texture

**a**
  Trial 2/10: Looking at the 5 images, I need to identify

{'word': 'tafno',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Looking at the 5 images, I need to identify what observable feature is common to all objects labeled "tafno":\n\nImage 1: LEGO tree model - has a trunk structure\nImage 2: Artistic piece with branches/twigs - has a trunk-like structure\nImage 3: Real tree with branches and leaves - has a trunk\nImage 4: Tree roots spreading on ground - part of a tree\'s trunk system\nImage 5: Solitary tree in a field - has a prominent trunk\n\nThe most consistent observable pattern across all images is the presence of a **tree** or tree-like structure with a trunk. \n\n- Not "building material" (b) - images 3 and 5 show living trees in nature\n- Not "wood" (c) - image 3 shows a living tree with green leaves\n- Not "plant" (d) - too broad; would include grass, flowers, etc. not shown\n- Not "bark" (e) - not all images focus on bark texture\n\n**a**',
  'Looking at the 5 images, I need to identify what

In [33]:
run_experiment(word_data = tafno_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with gpt4 (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'tafno',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [34]:
run_experiment(word_data = tafno_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'tafno'...

Running 10 iterations for 'tafno' with gpt4 (elaborate prompt)...
  Trial 1/10: c) wood
  Trial 2/10: a
  Trial 3/10: c) wood
  Trial 4/10: c
  Trial 5/10: a
  Trial 6/10: c
  Trial 7/10: c) wood
  Trial 8/10: c) wood
  Trial 9/10: a
  Trial 10/10: c


{'word': 'tafno',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['c) wood',
  'a',
  'c) wood',
  'c',
  'a',
  'c',
  'c) wood',
  'c) wood',
  'a',
  'c']}

In [10]:
kermid_data = create_word_data(word ='kermid', image_paths =  ['main_quine_file/quine_images/rain_images/meteor_shower.jpg', 
                                              'main_quine_file/quine_images/rain_images/rain.jpg',
                                             'main_quine_file/quine_images/rain_images/rain_from_car_window.jpg',
                                             'main_quine_file/quine_images/rain_images/rain_with_rainbow.jpg',
                                             'main_quine_file/quine_images/rain_images/rainstorm.jpg'], options = {'a': 'rain',
                                                                                                                    'b': 'drizzle',
                                                                                                                    'c': 'humidity',
                                                                                                                    'd': 'precipitation',
                                                                                                                    'e': 'water'})

run_experiment(word_data = kermid_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with gemini (minimal prompt)...
  Trial 1/10: a)
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: The word "kermid" needs to be understood in the context of all five images.

1.  **Image 1:** Shows numerous meteors (shooting stars) streaking across the night sky. Meteors are particles falling from space into Earth's atmosphere.
2.  **Image 2:** Shows heavy rain falling on a street at night, with water splashing. Rain is a form of water falling from the sky.
3.  **Image 3:** Shows raindrops on a car window, indicating rain. Rain is a form of water falling from the sky.
4.  **Image 4:** Shows a rainbow and raindrops on a window, implying rain. Rain is a form of water falling from the sky.
5.  **Image 5:** Shows a thunderstorm with lightning and heavy rain. Rain is a form of water falling from the sky.

Let's evaluate the options:
*   **a) rain:** Fits images 2, 3, 4, 5 perfectly, but not image 1 (meteors a

{'word': 'kermid',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a)',
  'a',
  'a',
  'a',
  'The word "kermid" needs to be understood in the context of all five images.\n\n1.  **Image 1:** Shows numerous meteors (shooting stars) streaking across the night sky. Meteors are particles falling from space into Earth\'s atmosphere.\n2.  **Image 2:** Shows heavy rain falling on a street at night, with water splashing. Rain is a form of water falling from the sky.\n3.  **Image 3:** Shows raindrops on a car window, indicating rain. Rain is a form of water falling from the sky.\n4.  **Image 4:** Shows a rainbow and raindrops on a window, implying rain. Rain is a form of water falling from the sky.\n5.  **Image 5:** Shows a thunderstorm with lightning and heavy rain. Rain is a form of water falling from the sky.\n\nLet\'s evaluate the options:\n*   **a) rain:** Fits images 2, 3, 4, 5 perfectly, but not image 1 (meteors are not rain).\n*   **b) drizzle:** A v

In [11]:
run_experiment(word_data = kermid_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with gemini (elaborate prompt)...
  Trial 1/10: Based purely on observable patterns in the images and without assuming any prior framework for categorizing reality:

1.  **Image 1 (Meteors):** Shows streaks of light falling from the night sky.
2.  **Image 2 (Heavy Rain):** Shows streaks of liquid (rain) falling and splashing.
3.  **Image 3 (Raindrops on Window):** Shows droplets and streaks of liquid (water) on a surface.
4.  **Image 4 (Rainbow with Drops):** Shows droplets and streaks of liquid (water) on a surface, and a rainbow which is a phenomenon of light interacting with water droplets in the atmosphere.
5.  **Image 5 (Stormy Rain):** Shows streaks of liquid (rain) falling and splashing.

Let's evaluate the options against all five images:

*   **a) rain:** Fits images 2, 3 (implied), 4 (implied), 5. Does **not** fit image 1 (meteors are not rain).
*   **b) drizzle:** Fits some light rain scenarios, but images

{'word': 'kermid',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Based purely on observable patterns in the images and without assuming any prior framework for categorizing reality:\n\n1.  **Image 1 (Meteors):** Shows streaks of light falling from the night sky.\n2.  **Image 2 (Heavy Rain):** Shows streaks of liquid (rain) falling and splashing.\n3.  **Image 3 (Raindrops on Window):** Shows droplets and streaks of liquid (water) on a surface.\n4.  **Image 4 (Rainbow with Drops):** Shows droplets and streaks of liquid (water) on a surface, and a rainbow which is a phenomenon of light interacting with water droplets in the atmosphere.\n5.  **Image 5 (Stormy Rain):** Shows streaks of liquid (rain) falling and splashing.\n\nLet\'s evaluate the options against all five images:\n\n*   **a) rain:** Fits images 2, 3 (implied), 4 (implied), 5. Does **not** fit image 1 (meteors are not rain).\n*   **b) drizzle:** Fits some light rain scenarios, but images 

In [13]:
run_experiment(word_data = kermid_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with claude (minimal prompt)...
  Trial 1/10: Looking at the 5 images:

1. A meteor shower/shooting stars in the night sky
2. Heavy rain on a street at night with traffic lights
3. Rain droplets on a car window
4. A rainbow visible through a wet window
5. Lightning storm with rain

All images show falling water from the sky in various forms - rain specifically appears in images 2-5, while image 1 shows meteors that create streaking trails similar to rain.

The common thread is **precipitation** - water falling from the atmosphere. This encompasses rain (which appears in most images), and is the broadest term that connects all the weather phenomena shown.

**d**
  Trial 2/10: Looking at all 5 images, I can see:
1. Meteor shower/shooting stars
2. Heavy rain on a street at night
3. Rain droplets on a car window
4. Rainbow through a wet window
5. Lightning storm with rain

The common element across all images is rain or 

{'word': 'kermid',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at the 5 images:\n\n1. A meteor shower/shooting stars in the night sky\n2. Heavy rain on a street at night with traffic lights\n3. Rain droplets on a car window\n4. A rainbow visible through a wet window\n5. Lightning storm with rain\n\nAll images show falling water from the sky in various forms - rain specifically appears in images 2-5, while image 1 shows meteors that create streaking trails similar to rain.\n\nThe common thread is **precipitation** - water falling from the atmosphere. This encompasses rain (which appears in most images), and is the broadest term that connects all the weather phenomena shown.\n\n**d**',
  'Looking at all 5 images, I can see:\n1. Meteor shower/shooting stars\n2. Heavy rain on a street at night\n3. Rain droplets on a car window\n4. Rainbow through a wet window\n5. Lightning storm with rain\n\nThe common element across all images is rain or fal

In [14]:
run_experiment(word_data = kermid_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with claude (elaborate prompt)...
  Trial 1/10: I need to observe what is common across all five images where "kermid" was used.

Image 1: Shows streaks across a night sky - these appear to be meteor trails or star trails
Image 2: Shows streaks with blurred lights, appears to be rain falling at night with city lights
Image 3: Shows water droplets on glass/window with streaks running down
Image 4: Shows a rainbow through a wet window with water droplets and streaks
Image 5: Shows lightning with visible rain streaks falling

The common observable pattern across all images is **visible streaks of falling water** - the linear trails that rain makes when falling. This is distinct from:
- Just "water" (which wouldn't require the streak pattern)
- "Humidity" (which is invisible)
- "Drizzle" (which typically has very fine, less visible drops)
- "Precipitation" (too general, includes snow, hail, etc.)

The specific pattern is

{'word': 'kermid',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['I need to observe what is common across all five images where "kermid" was used.\n\nImage 1: Shows streaks across a night sky - these appear to be meteor trails or star trails\nImage 2: Shows streaks with blurred lights, appears to be rain falling at night with city lights\nImage 3: Shows water droplets on glass/window with streaks running down\nImage 4: Shows a rainbow through a wet window with water droplets and streaks\nImage 5: Shows lightning with visible rain streaks falling\n\nThe common observable pattern across all images is **visible streaks of falling water** - the linear trails that rain makes when falling. This is distinct from:\n- Just "water" (which wouldn\'t require the streak pattern)\n- "Humidity" (which is invisible)\n- "Drizzle" (which typically has very fine, less visible drops)\n- "Precipitation" (too general, includes snow, hail, etc.)\n\nThe specific pattern 

In [15]:
run_experiment(word_data = kermid_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with gpt4 (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: c) humidity
  Trial 3/10: c) humidity
  Trial 4/10: e) water
  Trial 5/10: d) precipitation
  Trial 6/10: b
  Trial 7/10: d) precipitation
  Trial 8/10: b) drizzle
  Trial 9/10: b) drizzle
  Trial 10/10: d) precipitation


{'word': 'kermid',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a',
  'c) humidity',
  'c) humidity',
  'e) water',
  'd) precipitation',
  'b',
  'd) precipitation',
  'b) drizzle',
  'b) drizzle',
  'd) precipitation']}

In [16]:
run_experiment(word_data = kermid_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'kermid'...

Running 10 iterations for 'kermid' with gpt4 (elaborate prompt)...
  Trial 1/10: d
  Trial 2/10: e) water
  Trial 3/10: d) precipitation
  Trial 4/10: d) precipitation
  Trial 5/10: d) precipitation
  Trial 6/10: d) precipitation
  Trial 7/10: d) precipitation
  Trial 8/10: d
  Trial 9/10: d
  Trial 10/10: e) water


{'word': 'kermid',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['d',
  'e) water',
  'd) precipitation',
  'd) precipitation',
  'd) precipitation',
  'd) precipitation',
  'd) precipitation',
  'd',
  'd',
  'e) water']}

In [19]:
voldak_data = create_word_data(word ='voldak', image_paths =  ['main_quine_file/quine_images/shadow_images/shadow.jpg', 
                                              'main_quine_file/quine_images/shadow_images/lamp_shadow.jpg',
                                             'main_quine_file/quine_images/shadow_images/shadow_conspiracy.jpg',
                                             'main_quine_file/quine_images/shadow_images/shadow_on_a_city.jpg',
                                             'main_quine_file/quine_images/shadow_images/shadow_monster.jpg'], options = {'a': 'shadow',
                                                                                                                    'b': 'shade',
                                                                                                                    'c': 'black',
                                                                                                                    'd': 'darkness',
                                                                                                                    'e': 'outline'})

run_experiment(word_data = voldak_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'voldak',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [21]:
run_experiment(word_data = voldak_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with gemini (elaborate prompt)...
  Trial 1/10: a)
  Trial 2/10: c
  Trial 3/10: a
  Trial 4/10: a)
  Trial 5/10: a)
  Trial 6/10: a)
  Trial 7/10: a
  Trial 8/10: a)
  Trial 9/10: a
  Trial 10/10: a


{'word': 'voldak',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a)', 'c', 'a', 'a)', 'a)', 'a)', 'a', 'a)', 'a', 'a']}

In [22]:
run_experiment(word_data = voldak_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with claude (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: Looking at the 5 images:

1. A shadow cast on a brick/stone wall
2. A lamp casting a shadow on a wall
3. Silhouettes of people visible through a window frame
4. A tall building casting a large shadow over a city
5. A blurred figure running in a dark alley

The common element across all images is **shadow** - either shadows cast by objects/people, or silhouettes (which are a type of shadow). The images show various types of shadows: cast shadows on surfaces, shadows of objects, human silhouettes, and architectural shadows.

**a**
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'voldak',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a',
  'Looking at the 5 images:\n\n1. A shadow cast on a brick/stone wall\n2. A lamp casting a shadow on a wall\n3. Silhouettes of people visible through a window frame\n4. A tall building casting a large shadow over a city\n5. A blurred figure running in a dark alley\n\nThe common element across all images is **shadow** - either shadows cast by objects/people, or silhouettes (which are a type of shadow). The images show various types of shadows: cast shadows on surfaces, shadows of objects, human silhouettes, and architectural shadows.\n\n**a**',
  'a',
  'a',
  'a',
  'a',
  'a',
  'a',
  'a',
  'a']}

In [23]:
run_experiment(word_data = voldak_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with claude (elaborate prompt)...
  Trial 1/10: I need to observe what is common across all 5 images where "voldak" was used.

Image 1: Shadow cast on a brick/stone wall
Image 2: Shadow of a desk lamp on a wall, with the lamp visible
Image 3: Silhouetted figures visible through/in a window frame
Image 4: Shadow cast by a tall building on surrounding cityscape
Image 5: A blurred, dark figure in motion in a street/alley

Common observable pattern: In each image, there is a dark shape or area that is caused by blocking light. These are all projected or cast dark forms created by objects interrupting light sources.

- Images 1, 2, and 4 clearly show shadows cast by objects
- Image 3 shows dark silhouettes (which function like shadows on a translucent surface)
- Image 5 is less clear but appears to show a dark figure, possibly in shadow or creating shadow

The pattern most strongly suggests these are all instances of shad

{'word': 'voldak',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['I need to observe what is common across all 5 images where "voldak" was used.\n\nImage 1: Shadow cast on a brick/stone wall\nImage 2: Shadow of a desk lamp on a wall, with the lamp visible\nImage 3: Silhouetted figures visible through/in a window frame\nImage 4: Shadow cast by a tall building on surrounding cityscape\nImage 5: A blurred, dark figure in motion in a street/alley\n\nCommon observable pattern: In each image, there is a dark shape or area that is caused by blocking light. These are all projected or cast dark forms created by objects interrupting light sources.\n\n- Images 1, 2, and 4 clearly show shadows cast by objects\n- Image 3 shows dark silhouettes (which function like shadows on a translucent surface)\n- Image 5 is less clear but appears to show a dark figure, possibly in shadow or creating shadow\n\nThe pattern most strongly suggests these are all instances of sha

In [24]:
run_experiment(word_data = voldak_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with gpt4 (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a) shadow
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'voldak',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a) shadow', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [25]:
run_experiment(word_data = voldak_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'voldak'...

Running 10 iterations for 'voldak' with gpt4 (elaborate prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a) shadow
  Trial 10/10: a


{'word': 'voldak',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a) shadow', 'a']}

In [29]:
nestru_data = create_word_data(word ='nestru', image_paths =  ['main_quine_file/quine_images/reflection_images/light_reflection.jpg', 
                                              'main_quine_file/quine_images/reflection_images/narcissus_myth.jpg',
                                             'main_quine_file/quine_images/reflection_images/reflection.jpg',
                                             'main_quine_file/quine_images/reflection_images/reflection_in_mirror.jpg',
                                             'main_quine_file/quine_images/reflection_images/reflection_in_water.jpg'], options = {'a': 'reflection',
                                                                                                                    'b': 'refract',
                                                                                                                    'c': 'symmetry',
                                                                                                                    'd': 'mirror',
                                                                                                                    'e': 'light'})

run_experiment(word_data = nestru_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a)
  Trial 4/10: a
  Trial 5/10: a)
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a)
  Trial 9/10: a
  Trial 10/10: a


{'word': 'nestru',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a)', 'a', 'a)', 'a', 'a', 'a)', 'a', 'a']}

In [30]:
run_experiment(word_data = nestru_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with gemini (elaborate prompt)...
  Trial 1/10: a
  Trial 2/10: The most consistent observable pattern across all five images is the phenomenon of light bouncing off a surface to create an image or a visible pattern.

Let's analyze each option:
a) **reflection**: In all five images, a reflection is present.
    - Image 1: Reflected light rays creating a pattern.
    - Image 2: A reflection of the statue in the water.
    - Image 3: A reflection of the man in the mirror.
    - Image 4: A reflection of the man's face in the phone screen.
    - Image 5: Reflections of the sky and trees in the water.
    This option fits all images perfectly, describing the common visual event or the resulting image.

b) **refract**: Refraction is the bending of light as it passes through a medium. While light interacts with water, the primary visual phenomenon displayed in these images is the bouncing back of light (reflection), not its

{'word': 'nestru',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a',
  'The most consistent observable pattern across all five images is the phenomenon of light bouncing off a surface to create an image or a visible pattern.\n\nLet\'s analyze each option:\na) **reflection**: In all five images, a reflection is present.\n    - Image 1: Reflected light rays creating a pattern.\n    - Image 2: A reflection of the statue in the water.\n    - Image 3: A reflection of the man in the mirror.\n    - Image 4: A reflection of the man\'s face in the phone screen.\n    - Image 5: Reflections of the sky and trees in the water.\n    This option fits all images perfectly, describing the common visual event or the resulting image.\n\nb) **refract**: Refraction is the bending of light as it passes through a medium. While light interacts with water, the primary visual phenomenon displayed in these images is the bouncing back of light (reflection), not its bending 

In [31]:
run_experiment(word_data = nestru_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with claude (minimal prompt)...
  Trial 1/10: Looking at all 5 images, I can see they share a common theme:

1. A mirror with light radiating outward
2. A statue reflected in water
3. A person standing in front of a mirror
4. A person's face reflected in a handheld mirror
5. Trees and sky reflected in water

The consistent element across all images is **reflection** - whether in mirrors or water surfaces.

**a**
  Trial 2/10: Looking at all 5 images, I can identify the common theme:

1. A mirror with light rays emanating from it
2. A statue reflected in water
3. A person standing before a mirror
4. A person's reflection in a handheld mirror
5. Trees and sky reflected in water

The consistent element across all images is **reflection** - whether in mirrors or water surfaces.

**a**
  Trial 3/10: Looking at all 5 images, I can see they share a common theme:

1. A mirror with light rays creating a starburst pattern
2. A

{'word': 'nestru',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ["Looking at all 5 images, I can see they share a common theme:\n\n1. A mirror with light radiating outward\n2. A statue reflected in water\n3. A person standing in front of a mirror\n4. A person's face reflected in a handheld mirror\n5. Trees and sky reflected in water\n\nThe consistent element across all images is **reflection** - whether in mirrors or water surfaces.\n\n**a**",
  "Looking at all 5 images, I can identify the common theme:\n\n1. A mirror with light rays emanating from it\n2. A statue reflected in water\n3. A person standing before a mirror\n4. A person's reflection in a handheld mirror\n5. Trees and sky reflected in water\n\nThe consistent element across all images is **reflection** - whether in mirrors or water surfaces.\n\n**a**",
  "Looking at all 5 images, I can see they share a common theme:\n\n1. A mirror with light rays creating a starburst pattern\n2. A statue 

In [32]:
run_experiment(word_data = nestru_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with claude (elaborate prompt)...
  Trial 1/10: Looking at the observable patterns across all 5 images:

Image 1: Light radiating from a central point through a framed mirror
Image 2: Statue reflected in water
Image 3: Person standing before a framed mirror
Image 4: Person's face shown in a handheld mirror
Image 5: Sky and trees reflected in water

The common observable element is a **surface that shows a reversed/duplicated image** of something. In images 1, 3, and 4, this surface is specifically a mirror (framed reflective surface). In images 2 and 5, water serves this function.

While "reflection" describes the phenomenon occurring, "mirror" describes the specific type of object/surface present or implied in all cases (actual mirrors in 1, 3, 4; water acting as a mirror in 2, 5).

**d**
  Trial 2/10: I need to observe what is common across all 5 images where "nestru" was used.

Image 1: A mirror with light radiati

{'word': 'nestru',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Looking at the observable patterns across all 5 images:\n\nImage 1: Light radiating from a central point through a framed mirror\nImage 2: Statue reflected in water\nImage 3: Person standing before a framed mirror\nImage 4: Person\'s face shown in a handheld mirror\nImage 5: Sky and trees reflected in water\n\nThe common observable element is a **surface that shows a reversed/duplicated image** of something. In images 1, 3, and 4, this surface is specifically a mirror (framed reflective surface). In images 2 and 5, water serves this function.\n\nWhile "reflection" describes the phenomenon occurring, "mirror" describes the specific type of object/surface present or implied in all cases (actual mirrors in 1, 3, 4; water acting as a mirror in 2, 5).\n\n**d**',
  'I need to observe what is common across all 5 images where "nestru" was used.\n\nImage 1: A mirror with light radiating from

In [33]:
run_experiment(word_data = nestru_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with gpt4 (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: d) mirror
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'nestru',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'd) mirror', 'a', 'a', 'a', 'a', 'a', 'a']}

In [34]:
run_experiment(word_data = nestru_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'nestru'...

Running 10 iterations for 'nestru' with gpt4 (elaborate prompt)...
  Trial 1/10: a) reflection
  Trial 2/10: a) reflection
  Trial 3/10: a) reflection
  Trial 4/10: a) reflection
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'nestru',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a) reflection',
  'a) reflection',
  'a) reflection',
  'a) reflection',
  'a',
  'a',
  'a',
  'a',
  'a',
  'a']}

In [40]:
pimkar_data = create_word_data(word ='pimkar', image_paths =  ['main_quine_file/quine_images/wave_images/wave.jpg', 
                                              'main_quine_file/quine_images/wave_images/ripple.jpg',
                                             'main_quine_file/quine_images/wave_images/aurora_borealis_waves.jpg',
                                             'main_quine_file/quine_images/wave_images/sound_wave.jpg',
                                             'main_quine_file/quine_images/wave_images/tsunami.jpg'], options = {'a': 'wave',
                                                                                                                    'b': 'flow',
                                                                                                                    'c': 'ocean',
                                                                                                                    'd': 'water',
                                                                                                                    'e': 'crest'})

run_experiment(word_data = pimkar_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'pimkar',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [41]:
run_experiment(word_data = pimkar_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with gemini (elaborate prompt)...
  Trial 1/10: a)
  Trial 2/10: a)
  Trial 3/10: a)
  Trial 4/10: a)
  Trial 5/10: a)
  Trial 6/10: a) wave
  Trial 7/10: a
  Trial 8/10: a)
  Trial 9/10: a
  Trial 10/10: a


{'word': 'pimkar',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a)', 'a)', 'a)', 'a)', 'a)', 'a) wave', 'a', 'a)', 'a', 'a']}

In [42]:
run_experiment(word_data = pimkar_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with claude (minimal prompt)...
  Trial 1/10: Looking at all 5 images, I can see:
1. A wave with splashing water
2. Ripples in water (concentric circles)
3. Northern lights with flowing patterns
4. Sound/light waves flowing
5. A large wave threatening a ship

The common element across all images is the concept of **flow** - water flowing in waves, ripples flowing outward, aurora flowing in the sky, energy flowing as waves, and the dynamic flowing movement of ocean waves.

**b**
  Trial 2/10: Looking at the 5 images:
1. A crashing ocean wave with spray
2. Ripples/circles in calm water
3. Aurora borealis with flowing patterns
4. Sound/energy waves visualization
5. A massive wave threatening a ship

The common element across all these images is the concept of **wave** - whether it's ocean waves, water ripples (wave patterns), aurora waves, sound waves, or sea waves.

**a**
  Trial 3/10: Looking at the 5 images, I can se

{'word': 'pimkar',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at all 5 images, I can see:\n1. A wave with splashing water\n2. Ripples in water (concentric circles)\n3. Northern lights with flowing patterns\n4. Sound/light waves flowing\n5. A large wave threatening a ship\n\nThe common element across all images is the concept of **flow** - water flowing in waves, ripples flowing outward, aurora flowing in the sky, energy flowing as waves, and the dynamic flowing movement of ocean waves.\n\n**b**',
  "Looking at the 5 images:\n1. A crashing ocean wave with spray\n2. Ripples/circles in calm water\n3. Aurora borealis with flowing patterns\n4. Sound/energy waves visualization\n5. A massive wave threatening a ship\n\nThe common element across all these images is the concept of **wave** - whether it's ocean waves, water ripples (wave patterns), aurora waves, sound waves, or sea waves.\n\n**a**",
  "Looking at the 5 images, I can see:\n1. A wave

In [43]:
run_experiment(word_data = pimkar_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with claude (elaborate prompt)...
  Trial 1/10: Looking at the observable patterns across all 5 images:

Image 1: Ocean wave with spray
Image 2: Water ripples in concentric circles
Image 3: Aurora borealis (atmospheric light phenomenon)
Image 4: Sound/energy waveforms
Image 5: Ocean wave near a ship

The common observable pattern is **wave-like motion or undulation** - whether in water, light, sound, or energy. This pattern appears across different media and contexts.

While water is present in some images, it's absent in images 3 and 4. "Flow" and "ocean" are too restrictive. "Crest" only applies to certain waves. The most consistent observable pattern is the wave phenomenon itself.

**e**
  Trial 2/10: Looking at the observable patterns across all 5 images:

Image 1: Ocean wave with spray
Image 2: Ripples in water
Image 3: Water body with aurora above
Image 4: Wave-like patterns (sound/energy visualization)
Image 5

{'word': 'pimkar',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Looking at the observable patterns across all 5 images:\n\nImage 1: Ocean wave with spray\nImage 2: Water ripples in concentric circles\nImage 3: Aurora borealis (atmospheric light phenomenon)\nImage 4: Sound/energy waveforms\nImage 5: Ocean wave near a ship\n\nThe common observable pattern is **wave-like motion or undulation** - whether in water, light, sound, or energy. This pattern appears across different media and contexts.\n\nWhile water is present in some images, it\'s absent in images 3 and 4. "Flow" and "ocean" are too restrictive. "Crest" only applies to certain waves. The most consistent observable pattern is the wave phenomenon itself.\n\n**e**',
  'Looking at the observable patterns across all 5 images:\n\nImage 1: Ocean wave with spray\nImage 2: Ripples in water\nImage 3: Water body with aurora above\nImage 4: Wave-like patterns (sound/energy visualization)\nImage 5: O

In [44]:
run_experiment(word_data = pimkar_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with gpt4 (minimal prompt)...
  Trial 1/10: a) wave
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a) wave
  Trial 10/10: a


{'word': 'pimkar',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a) wave', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a) wave', 'a']}

In [45]:
run_experiment(word_data = pimkar_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'pimkar'...

Running 10 iterations for 'pimkar' with gpt4 (elaborate prompt)...
  Trial 1/10: b
  Trial 2/10: a
  Trial 3/10: b
  Trial 4/10: b
  Trial 5/10: b) flow
  Trial 6/10: b
  Trial 7/10: b
  Trial 8/10: b
  Trial 9/10: b) flow
  Trial 10/10: b


{'word': 'pimkar',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['b', 'a', 'b', 'b', 'b) flow', 'b', 'b', 'b', 'b) flow', 'b']}

In [47]:
zaldro_data = create_word_data(word ='zaldro', image_paths =  ['main_quine_file/quine_images/cloud_images/clouds.jpg', 
                                              'main_quine_file/quine_images/cloud_images/fog.jpg',
                                             'main_quine_file/quine_images/cloud_images/airplane_clouds.jpg',
                                             'main_quine_file/quine_images/cloud_images/thunder_clouds.jpg',
                                             'main_quine_file/quine_images/cloud_images/volcano_cloud.jpg'], options = {'a': 'clouds',
                                                                                                                    'b': 'scudding',
                                                                                                                    'c': 'white',
                                                                                                                    'd': 'vapor',
                                                                                                                    'e': 'smoke'})

run_experiment(word_data = zaldro_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a)
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: The images display various atmospheric phenomena:
1.  **Image 1:** Clear sky with numerous cumulus and stratocumulus clouds.
2.  **Image 2:** A foggy or misty landscape, which is essentially a cloud at ground level.
3.  **Image 3:** View from an airplane window, showing scattered clouds below.
4.  **Image 4:** A thunderstorm with dark, heavy storm clouds and lightning.
5.  **Image 5:** A volcanic eruption producing a large plume of smoke and ash, which forms an ash cloud, along with other clouds in the background.

Let's evaluate the options:
*   **a) clouds:** This term directly describes the main feature in images 1, 3, and 4. Fog/mist (image 2) are types of clouds. The volcanic plume (image 5) is commonly referred to as an ash cloud or smoke 

{'word': 'zaldro',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a',
  'a',
  'a',
  'a',
  'a',
  'a)',
  'a',
  'a',
  'The images display various atmospheric phenomena:\n1.  **Image 1:** Clear sky with numerous cumulus and stratocumulus clouds.\n2.  **Image 2:** A foggy or misty landscape, which is essentially a cloud at ground level.\n3.  **Image 3:** View from an airplane window, showing scattered clouds below.\n4.  **Image 4:** A thunderstorm with dark, heavy storm clouds and lightning.\n5.  **Image 5:** A volcanic eruption producing a large plume of smoke and ash, which forms an ash cloud, along with other clouds in the background.\n\nLet\'s evaluate the options:\n*   **a) clouds:** This term directly describes the main feature in images 1, 3, and 4. Fog/mist (image 2) are types of clouds. The volcanic plume (image 5) is commonly referred to as an ash cloud or smoke cloud. This option fits all images well.\n*   **b) scudding:** This describe

In [48]:
run_experiment(word_data = zaldro_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with gemini (elaborate prompt)...
  Trial 1/10: A
  Trial 2/10: a) clouds
  Trial 3/10: A
  Trial 4/10: A
  Trial 5/10: a)
  Trial 6/10: a
  Trial 7/10: a) clouds
  Trial 8/10: a
  Trial 9/10: a)
  Trial 10/10: a) clouds


{'word': 'zaldro',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['A',
  'a) clouds',
  'A',
  'A',
  'a)',
  'a',
  'a) clouds',
  'a',
  'a)',
  'a) clouds']}

In [50]:
run_experiment(word_data = zaldro_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with claude (minimal prompt)...
  Trial 1/10: Looking at all 5 images, I can see they all prominently feature clouds or cloud-like formations:

1. Dramatic clouds over a rural landscape
2. Misty/foggy scene with trees (fog is essentially ground-level clouds)
3. View from airplane window showing clouds
4. Storm clouds with lightning
5. Volcanic ash/smoke cloud (which behaves like clouds)

The common element across all images is clouds in various forms.

**a**
  Trial 2/10: Looking at all 5 images, I can see they all feature prominent cloud formations or atmospheric phenomena:

1. Dramatic clouds in a blue sky over countryside
2. Misty/foggy clouds around trees at dawn/dusk
3. Clouds viewed from an airplane window
4. Storm clouds with lightning
5. Volcanic ash/smoke plume

The common element across all images is **clouds** (or cloud-like formations including fog, storm clouds, and volcanic plumes).

**a**
  Trial 3/10:

{'word': 'zaldro',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at all 5 images, I can see they all prominently feature clouds or cloud-like formations:\n\n1. Dramatic clouds over a rural landscape\n2. Misty/foggy scene with trees (fog is essentially ground-level clouds)\n3. View from airplane window showing clouds\n4. Storm clouds with lightning\n5. Volcanic ash/smoke cloud (which behaves like clouds)\n\nThe common element across all images is clouds in various forms.\n\n**a**',
  'Looking at all 5 images, I can see they all feature prominent cloud formations or atmospheric phenomena:\n\n1. Dramatic clouds in a blue sky over countryside\n2. Misty/foggy clouds around trees at dawn/dusk\n3. Clouds viewed from an airplane window\n4. Storm clouds with lightning\n5. Volcanic ash/smoke plume\n\nThe common element across all images is **clouds** (or cloud-like formations including fog, storm clouds, and volcanic plumes).\n\n**a**',
  'Looking at

In [51]:
run_experiment(word_data = zaldro_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with claude (elaborate prompt)...
  Trial 1/10: Looking at the five images, I need to identify what visible element appears in all of them:

Image 1: Clouds in the sky
Image 2: Fog/mist among trees
Image 3: Clouds viewed from airplane
Image 4: Storm clouds with lightning
Image 5: Volcanic ash/smoke plume

The common observable pattern across all images is the presence of suspended particles in the air - whether water droplets (clouds, fog) or ash particles (volcanic plume). This points to **vapor** as the unifying concept, since it encompasses all these atmospheric phenomena as suspended matter in air.

While "clouds" might seem obvious, image 2 shows fog (which is technically a cloud at ground level but might not be categorized the same way), and image 5 shows volcanic emissions which are distinctly not water clouds. "Vapor" as a broader category of "substance suspended in air" fits all five cases.

**d**
  Trial 2/

{'word': 'zaldro',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Looking at the five images, I need to identify what visible element appears in all of them:\n\nImage 1: Clouds in the sky\nImage 2: Fog/mist among trees\nImage 3: Clouds viewed from airplane\nImage 4: Storm clouds with lightning\nImage 5: Volcanic ash/smoke plume\n\nThe common observable pattern across all images is the presence of suspended particles in the air - whether water droplets (clouds, fog) or ash particles (volcanic plume). This points to **vapor** as the unifying concept, since it encompasses all these atmospheric phenomena as suspended matter in air.\n\nWhile "clouds" might seem obvious, image 2 shows fog (which is technically a cloud at ground level but might not be categorized the same way), and image 5 shows volcanic emissions which are distinctly not water clouds. "Vapor" as a broader category of "substance suspended in air" fits all five cases.\n\n**d**',
  'I need

In [52]:
run_experiment(word_data = zaldro_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with gpt4 (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: c) white
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: e
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: e


{'word': 'zaldro',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'c) white', 'a', 'a', 'e', 'a', 'a', 'a', 'e']}

In [53]:
run_experiment(word_data = zaldro_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'zaldro'...

Running 10 iterations for 'zaldro' with gpt4 (elaborate prompt)...
  Trial 1/10: e) smoke
  Trial 2/10: e) smoke
  Trial 3/10: e
  Trial 4/10: e
  Trial 5/10: e
  Trial 6/10: c) white
  Trial 7/10: e) smoke
  Trial 8/10: e) smoke
  Trial 9/10: e
  Trial 10/10: e


{'word': 'zaldro',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['e) smoke',
  'e) smoke',
  'e',
  'e',
  'e',
  'c) white',
  'e) smoke',
  'e) smoke',
  'e',
  'e']}

In [62]:
grifon_data = create_word_data(word ='grifon', image_paths =  ['main_quine_file/quine_images/stop_sign_images/old_stop_sign.jpg', 
                                              'main_quine_file/quine_images/stop_sign_images/stop_sign.jpg',
                                             'main_quine_file/quine_images/stop_sign_images/stop_zoomed_in.jpg',
                                             'main_quine_file/quine_images/stop_sign_images/painting_stop_sign.jpg',
                                             'main_quine_file/quine_images/stop_sign_images/red_surface_stop_sign.jpg'], options = {'a': 'stop sign',
                                                                                                                    'b': 'stopping',
                                                                                                                    'c': 'octagon',
                                                                                                                    'd': 'traffic sign',
                                                                                                                    'e': 'letters'})

run_experiment(word_data = grifon_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a)
  Trial 7/10: a
  Trial 8/10: a)
  Trial 9/10: a
  Trial 10/10: a)


{'word': 'grifon',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a)', 'a', 'a)', 'a', 'a)']}

In [63]:
run_experiment(word_data = grifon_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with gemini (elaborate prompt)...
  Trial 1/10: c)
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a)
  Trial 5/10: a)
  Trial 6/10: c
  Trial 7/10: a
  Trial 8/10: a)
  Trial 9/10: c)
  Trial 10/10: c)


{'word': 'grifon',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['c)', 'a', 'a', 'a)', 'a)', 'c', 'a', 'a)', 'c)', 'c)']}

In [64]:
run_experiment(word_data = grifon_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with claude (minimal prompt)...
  Trial 1/10: c

The word "grifon" refers to the octagonal shape of the stop sign visible in all the images.
  Trial 2/10: c

The word "grifon" refers to the octagonal shape of the stop sign visible in all 5 images.
  Trial 3/10: c

The word "grifon" refers to the octagonal shape of the stop sign visible in all five images.
  Trial 4/10: c

The word "grifon" refers to the octagonal shape of the stop sign visible in all the images.
  Trial 5/10: Looking at the 5 images, I can see:
- Image 1: A weathered, peeling stop sign
- Image 2: A clean stop sign
- Image 3: Close-up of peeling paint on a stop sign
- Image 4: A stop sign in a desert landscape
- Image 5: Close-up of the edge/corner of a stop sign

The common element across all images is the octagonal shape of the stop sign itself. While these are stop signs, the word "grifon" appears to refer to the geometric shape rather than the sig

{'word': 'grifon',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['c\n\nThe word "grifon" refers to the octagonal shape of the stop sign visible in all the images.',
  'c\n\nThe word "grifon" refers to the octagonal shape of the stop sign visible in all 5 images.',
  'c\n\nThe word "grifon" refers to the octagonal shape of the stop sign visible in all five images.',
  'c\n\nThe word "grifon" refers to the octagonal shape of the stop sign visible in all the images.',
  'Looking at the 5 images, I can see:\n- Image 1: A weathered, peeling stop sign\n- Image 2: A clean stop sign\n- Image 3: Close-up of peeling paint on a stop sign\n- Image 4: A stop sign in a desert landscape\n- Image 5: Close-up of the edge/corner of a stop sign\n\nThe common element across all images is the octagonal shape of the stop sign itself. While these are stop signs, the word "grifon" appears to refer to the geometric shape rather than the sign\'s function or the letters on it

In [65]:
run_experiment(word_data = grifon_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with claude (elaborate prompt)...
  Trial 1/10: c
  Trial 2/10: c
  Trial 3/10: c
  Trial 4/10: c
  Trial 5/10: c
  Trial 6/10: c
  Trial 7/10: c
  Trial 8/10: c
  Trial 9/10: c
  Trial 10/10: c


{'word': 'grifon',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['c', 'c', 'c', 'c', 'c', 'c', 'c', 'c', 'c', 'c']}

In [66]:
run_experiment(word_data = grifon_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with gpt4 (minimal prompt)...
  Trial 1/10: a) stop sign
  Trial 2/10: a) stop sign
  Trial 3/10: a) stop sign
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a) stop sign
  Trial 8/10: a) stop sign
  Trial 9/10: a) stop sign
  Trial 10/10: a) stop sign


{'word': 'grifon',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a) stop sign',
  'a) stop sign',
  'a) stop sign',
  'a',
  'a',
  'a',
  'a) stop sign',
  'a) stop sign',
  'a) stop sign',
  'a) stop sign']}

In [67]:
run_experiment(word_data = grifon_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'grifon'...

Running 10 iterations for 'grifon' with gpt4 (elaborate prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: c) octagon
  Trial 4/10: a) stop sign
  Trial 5/10: c) octagon
  Trial 6/10: a
  Trial 7/10: c) octagon
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: c) octagon


{'word': 'grifon',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a',
  'a',
  'c) octagon',
  'a) stop sign',
  'c) octagon',
  'a',
  'c) octagon',
  'a',
  'a',
  'c) octagon']}

In [68]:
lukima_data = create_word_data(word ='lukima', image_paths =  ['main_quine_file/quine_images/lemon_images/lemon.jpg', 
                                              'main_quine_file/quine_images/lemon_images/lemon_in_bowl.jpg',
                                             'main_quine_file/quine_images/lemon_images/lemon_rind.jpg',
                                             'main_quine_file/quine_images/lemon_images/peeled_lemon.jpg',
                                             'main_quine_file/quine_images/lemon_images/toy_lemon.jpg'], options = {'a': 'lemon',
                                                                                                                    'b': 'sour',
                                                                                                                    'c': 'yellow',
                                                                                                                    'd': 'citrus',
                                                                                                                    'e': 'rind'})

run_experiment(word_data = lukima_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'lukima',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [69]:
run_experiment(word_data = lukima_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with gemini (elaborate prompt)...
  Trial 1/10: a
  Trial 2/10: a)
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: e
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a)
  Trial 9/10: a
  Trial 10/10: a


{'word': 'lukima',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a', 'a)', 'a', 'a', 'e', 'a', 'a', 'a)', 'a', 'a']}

In [70]:
run_experiment(word_data = lukima_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with claude (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: Looking at the 5 images:
1. A single lemon
2. A lemon in a fruit bowl with other fruits
3. Close-up of lemon texture/skin
4. A cut lemon showing the inside with peel
5. A lemon among various other fruits

The common element in all images is the presence of a lemon specifically. While lemons are yellow, sour, citrus fruits with rinds, the word "lukima" appears consistently with the lemon itself across all contexts.

**a**
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'lukima',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a',
  'a',
  'a',
  'Looking at the 5 images:\n1. A single lemon\n2. A lemon in a fruit bowl with other fruits\n3. Close-up of lemon texture/skin\n4. A cut lemon showing the inside with peel\n5. A lemon among various other fruits\n\nThe common element in all images is the presence of a lemon specifically. While lemons are yellow, sour, citrus fruits with rinds, the word "lukima" appears consistently with the lemon itself across all contexts.\n\n**a**',
  'a',
  'a',
  'a',
  'a',
  'a',
  'a']}

In [71]:
run_experiment(word_data = lukima_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with claude (elaborate prompt)...
  Trial 1/10: I need to analyze what is consistently present across all 5 images where "lukima" was used.

Image 1: A single lemon on a dark surface
Image 2: A wooden bowl with fruits including a lemon, apples, and oranges
Image 3: Close-up of a lemon's textured surface
Image 4: A cut lemon showing the interior with peel strips
Image 5: Various fruits including a lemon, apples, pear, and grapes

Observable pattern: In every image, there is specifically a lemon present. The lemon is the only object that appears in all 5 situations.

- "Sour" (b) cannot be observed visually
- "Yellow" (c) - while lemons are yellow, other yellow objects appear (like the pear in image 5), and not all yellow things are pointed at
- "Citrus" (d) - there are oranges in image 2, but they weren't necessarily what was pointed at
- "Rind" (e) - only clearly visible in image 4, not all images

The most consisten

{'word': 'lukima',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['I need to analyze what is consistently present across all 5 images where "lukima" was used.\n\nImage 1: A single lemon on a dark surface\nImage 2: A wooden bowl with fruits including a lemon, apples, and oranges\nImage 3: Close-up of a lemon\'s textured surface\nImage 4: A cut lemon showing the interior with peel strips\nImage 5: Various fruits including a lemon, apples, pear, and grapes\n\nObservable pattern: In every image, there is specifically a lemon present. The lemon is the only object that appears in all 5 situations.\n\n- "Sour" (b) cannot be observed visually\n- "Yellow" (c) - while lemons are yellow, other yellow objects appear (like the pear in image 5), and not all yellow things are pointed at\n- "Citrus" (d) - there are oranges in image 2, but they weren\'t necessarily what was pointed at\n- "Rind" (e) - only clearly visible in image 4, not all images\n\nThe most consi

In [72]:
run_experiment(word_data = lukima_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with gpt4 (minimal prompt)...
  Trial 1/10: a) lemon
  Trial 2/10: a
  Trial 3/10: a) lemon
  Trial 4/10: a) lemon
  Trial 5/10: a) lemon
  Trial 6/10: a) lemon
  Trial 7/10: a) lemon
  Trial 8/10: a
  Trial 9/10: a) lemon
  Trial 10/10: a


{'word': 'lukima',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a) lemon',
  'a',
  'a) lemon',
  'a) lemon',
  'a) lemon',
  'a) lemon',
  'a) lemon',
  'a',
  'a) lemon',
  'a']}

In [73]:
run_experiment(word_data = lukima_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'lukima'...

Running 10 iterations for 'lukima' with gpt4 (elaborate prompt)...
  Trial 1/10: c) yellow
  Trial 2/10: a) lemon
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a) lemon
  Trial 8/10: a
  Trial 9/10: c) yellow
  Trial 10/10: a


{'word': 'lukima',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['c) yellow',
  'a) lemon',
  'a',
  'a',
  'a',
  'a',
  'a) lemon',
  'a',
  'c) yellow',
  'a']}

In [74]:
torvel_data = create_word_data(word ='torvel', image_paths =  ['main_quine_file/quine_images/brick_images/brick.jpg', 
                                              'main_quine_file/quine_images/brick_images/brick_fireplace.jpg',
                                             'main_quine_file/quine_images/brick_images/brick_surface.jpg',
                                             'main_quine_file/quine_images/brick_images/many_bricks.jpg',
                                             'main_quine_file/quine_images/brick_images/statue_of_bricks.jpg'], options = {'a': 'brick',
                                                                                                                    'b': 'building',
                                                                                                                    'c': 'red',
                                                                                                                    'd': 'stone',
                                                                                                                    'e': 'rectangle'})

run_experiment(word_data = torvel_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a)
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'torvel',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a)', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [75]:
run_experiment(word_data = torvel_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with gemini (elaborate prompt)...
  Trial 1/10: a)
  Trial 2/10: Based ONLY on observable patterns in the images, let's analyze each option:

*   **a) brick:**
    *   Image 1: A single brick. (Fits)
    *   Image 2: A fireplace made of bricks. (Fits for the components)
    *   Image 3: A single brick. (Fits)
    *   Image 4: A wall made of bricks. (Fits for the components)
    *   Image 5: A sculpture made of bricks. (Fits for the components)
    *   This option is consistent across all situations, referring to the fundamental object or the constituent material.

*   **b) building:**
    *   Image 1: A single brick, not a building. (Fails)
    *   Image 2: A fireplace, part of a building, but not a whole building. (Weak fit)
    *   Image 3: A single brick, not a building. (Fails)
    *   Image 4: A wall, part of a building, but not a whole building. (Weak fit)
    *   Image 5: A sculpture, not a building. (Fails)
 

{'word': 'torvel',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a)',
  'Based ONLY on observable patterns in the images, let\'s analyze each option:\n\n*   **a) brick:**\n    *   Image 1: A single brick. (Fits)\n    *   Image 2: A fireplace made of bricks. (Fits for the components)\n    *   Image 3: A single brick. (Fits)\n    *   Image 4: A wall made of bricks. (Fits for the components)\n    *   Image 5: A sculpture made of bricks. (Fits for the components)\n    *   This option is consistent across all situations, referring to the fundamental object or the constituent material.\n\n*   **b) building:**\n    *   Image 1: A single brick, not a building. (Fails)\n    *   Image 2: A fireplace, part of a building, but not a whole building. (Weak fit)\n    *   Image 3: A single brick, not a building. (Fails)\n    *   Image 4: A wall, part of a building, but not a whole building. (Weak fit)\n    *   Image 5: A sculpture, not a building. (Fails)\n    * 

In [76]:
run_experiment(word_data = torvel_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with claude (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'torvel',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [77]:
run_experiment(word_data = torvel_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with claude (elaborate prompt)...
  Trial 1/10: Looking at the images, I need to identify what common observable feature appears in all 5 situations where "torvel" was used.

Image 1: A single brick on pavement
Image 2: A brick fireplace structure
Image 3: Stacked bricks (close-up)
Image 4: A brick wall
Image 5: A human figure sculpture made of bricks

Common observable patterns:
- All images contain bricks (the rectangular clay/ceramic building blocks)
- Not all images show complete buildings (images 1, 3 don't)
- Colors vary (orange, red, gray, tan tones)
- Materials are all the same manufactured brick units
- Rectangle shapes appear but are specifically in the form of bricks

The most consistent observable pattern across all 5 images is the presence of bricks as objects.

**a**
  Trial 2/10: Looking at the images, I need to identify what common observable feature appears in all 5 situations:

Image 1: A single bri

{'word': 'torvel',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Looking at the images, I need to identify what common observable feature appears in all 5 situations where "torvel" was used.\n\nImage 1: A single brick on pavement\nImage 2: A brick fireplace structure\nImage 3: Stacked bricks (close-up)\nImage 4: A brick wall\nImage 5: A human figure sculpture made of bricks\n\nCommon observable patterns:\n- All images contain bricks (the rectangular clay/ceramic building blocks)\n- Not all images show complete buildings (images 1, 3 don\'t)\n- Colors vary (orange, red, gray, tan tones)\n- Materials are all the same manufactured brick units\n- Rectangle shapes appear but are specifically in the form of bricks\n\nThe most consistent observable pattern across all 5 images is the presence of bricks as objects.\n\n**a**',
  'Looking at the images, I need to identify what common observable feature appears in all 5 situations:\n\nImage 1: A single brick

In [78]:
run_experiment(word_data = torvel_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with gpt4 (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a) brick
  Trial 3/10: a
  Trial 4/10: a) brick
  Trial 5/10: a) brick
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a) brick


{'word': 'torvel',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a',
  'a) brick',
  'a',
  'a) brick',
  'a) brick',
  'a',
  'a',
  'a',
  'a',
  'a) brick']}

In [79]:
run_experiment(word_data = torvel_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'torvel'...

Running 10 iterations for 'torvel' with gpt4 (elaborate prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a) brick
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a) brick
  Trial 10/10: a


{'word': 'torvel',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a) brick', 'a', 'a', 'a', 'a', 'a) brick', 'a']}

In [80]:
baskid_data = create_word_data(word ='baskid', image_paths =  ['main_quine_file/quine_images/grass_images/grass.jpg', 
                                              'main_quine_file/quine_images/grass_images/blade_of_grass.jpg',
                                             'main_quine_file/quine_images/grass_images/meadow.jpg',
                                             'main_quine_file/quine_images/grass_images/football_field_grass.jpg',
                                             'main_quine_file/quine_images/grass_images/garden_grass.jpg'], options = {'a': 'grass',
                                                                                                                    'b': 'planting',
                                                                                                                    'c': 'green',
                                                                                                                    'd': 'nature',
                                                                                                                    'e': 'chlorophyll'})

run_experiment(word_data = baskid_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with gemini (minimal prompt)...
  Trial 1/10: a)
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'baskid',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a)', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [81]:
run_experiment(word_data = baskid_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with gemini (elaborate prompt)...
  Trial 1/10: a)
  Trial 2/10: a
  Trial 3/10: a)
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a)
  Trial 7/10: a)
  Trial 8/10: a) grass
  Trial 9/10: a) grass
  Trial 10/10: a)


{'word': 'baskid',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a)',
  'a',
  'a)',
  'a',
  'a',
  'a)',
  'a)',
  'a) grass',
  'a) grass',
  'a)']}

In [83]:
run_experiment(word_data = baskid_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with claude (minimal prompt)...
  Trial 1/10: Looking at the 5 images, I can see:
1. A prairie/grassland with wild grasses
2. A close-up of a single grass blade
3. A mountain landscape with grassy meadows
4. A sports field with turf grass
5. A garden lawn with grass

The common element across all images is **grass** - whether wild, cultivated, on a sports field, or in a garden.

**a**
  Trial 2/10: Looking at the 5 images, I can see:
1. Wild grassland/prairie with grass
2. Close-up of a single grass blade
3. Mountain landscape with grassy meadow
4. Sports field with grass/turf
5. Garden lawn with grass

The common element across all images is **grass**.

**a**
  Trial 3/10: Looking at the 5 images, I can see:
1. Prairie/grassland with grass
2. Close-up of a single grass blade
3. Mountain landscape with grass-covered hills
4. Sports field with grass turf
5. Garden with grass lawn

The common element in all images is *

{'word': 'baskid',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at the 5 images, I can see:\n1. A prairie/grassland with wild grasses\n2. A close-up of a single grass blade\n3. A mountain landscape with grassy meadows\n4. A sports field with turf grass\n5. A garden lawn with grass\n\nThe common element across all images is **grass** - whether wild, cultivated, on a sports field, or in a garden.\n\n**a**',
  'Looking at the 5 images, I can see:\n1. Wild grassland/prairie with grass\n2. Close-up of a single grass blade\n3. Mountain landscape with grassy meadow\n4. Sports field with grass/turf\n5. Garden lawn with grass\n\nThe common element across all images is **grass**.\n\n**a**',
  'Looking at the 5 images, I can see:\n1. Prairie/grassland with grass\n2. Close-up of a single grass blade\n3. Mountain landscape with grass-covered hills\n4. Sports field with grass turf\n5. Garden with grass lawn\n\nThe common element in all images is **grass

In [84]:
run_experiment(word_data = baskid_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with claude (elaborate prompt)...
  Trial 1/10: Looking at the 5 images, I need to identify what observable element is common across all of them.

Image 1: Prairie/grassland with wild grasses
Image 2: Close-up of a single grass blade
Image 3: Mountain landscape with grassy meadow
Image 4: Tennis/sports court with artificial turf
Image 5: Garden with lawn grass

The most consistent observable pattern across all images is the presence of **grass** (whether natural wild grass, meadow grass, artificial turf, or lawn grass).

- "Green" (c) appears in all images but isn't the primary focus in image 3
- "Nature" (d) doesn't fit image 4 (artificial sports field)
- "Planting" (b) and "chlorophyll" (e) aren't directly observable patterns
- "Grass" (a) or grass-like surfaces are present in every single image

**a**
  Trial 2/10: Looking at the 5 images, I need to identify what observable element is common across all of them.

I

{'word': 'baskid',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Looking at the 5 images, I need to identify what observable element is common across all of them.\n\nImage 1: Prairie/grassland with wild grasses\nImage 2: Close-up of a single grass blade\nImage 3: Mountain landscape with grassy meadow\nImage 4: Tennis/sports court with artificial turf\nImage 5: Garden with lawn grass\n\nThe most consistent observable pattern across all images is the presence of **grass** (whether natural wild grass, meadow grass, artificial turf, or lawn grass).\n\n- "Green" (c) appears in all images but isn\'t the primary focus in image 3\n- "Nature" (d) doesn\'t fit image 4 (artificial sports field)\n- "Planting" (b) and "chlorophyll" (e) aren\'t directly observable patterns\n- "Grass" (a) or grass-like surfaces are present in every single image\n\n**a**',
  'Looking at the 5 images, I need to identify what observable element is common across all of them.\n\nIma

In [85]:
run_experiment(word_data = baskid_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with gpt4 (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a) grass
  Trial 3/10: a) grass
  Trial 4/10: a) grass
  Trial 5/10: a) grass
  Trial 6/10: a) grass
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'baskid',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a',
  'a) grass',
  'a) grass',
  'a) grass',
  'a) grass',
  'a) grass',
  'a',
  'a',
  'a',
  'a']}

In [86]:
run_experiment(word_data = baskid_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'baskid'...

Running 10 iterations for 'baskid' with gpt4 (elaborate prompt)...
  Trial 1/10: c) green
  Trial 2/10: c) green
  Trial 3/10: c) green
  Trial 4/10: c
  Trial 5/10: c
  Trial 6/10: c) green
  Trial 7/10: c) green
  Trial 8/10: c) green
  Trial 9/10: c) green
  Trial 10/10: c) green


{'word': 'baskid',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['c) green',
  'c) green',
  'c) green',
  'c',
  'c',
  'c) green',
  'c) green',
  'c) green',
  'c) green',
  'c) green']}

In [87]:
merplin_data = create_word_data(word ='merplin', image_paths =  ['main_quine_file/quine_images/snow_images/snow.jpg', 
                                              'main_quine_file/quine_images/snow_images/artificial_snow.jpg',
                                             'main_quine_file/quine_images/snow_images/dirty_snow.jpg',
                                             'main_quine_file/quine_images/snow_images/snowflake.jpg',
                                             'main_quine_file/quine_images/snow_images/mountain_snow.jpg'], options = {'a': 'snow',
                                                                                                                    'b': 'blizzard',
                                                                                                                    'c': 'white',
                                                                                                                    'd': 'winter',
                                                                                                                    'e': 'snowflakes'})

run_experiment(word_data = merplin_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'minimal')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with gemini (minimal prompt)...
  Trial 1/10: a
  Trial 2/10: a
  Trial 3/10: a
  Trial 4/10: a
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a
  Trial 8/10: a
  Trial 9/10: a
  Trial 10/10: a


{'word': 'merplin',
 'model': 'gemini',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']}

In [88]:
run_experiment(word_data = merplin_data, model_name = 'gemini', api_key = google_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with gemini (elaborate prompt)...
  Trial 1/10: a)
  Trial 2/10: a)
  Trial 3/10: a
  Trial 4/10: a)
  Trial 5/10: a)
  Trial 6/10: a)
  Trial 7/10: a)
  Trial 8/10: a)
  Trial 9/10: a
  Trial 10/10: a


{'word': 'merplin',
 'model': 'gemini',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a)', 'a)', 'a', 'a)', 'a)', 'a)', 'a)', 'a)', 'a', 'a']}

In [89]:
run_experiment(word_data = merplin_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'minimal')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with claude (minimal prompt)...
  Trial 1/10: Looking at the 5 images, I can see:

1. A frost-covered tree in a snowy winter landscape
2. A snow-making machine spraying snow on a ski slope
3. Snow/ice buildup along a roadside
4. A close-up macro shot of a single snowflake crystal
5. Snow-covered mountain peaks

The common element that appears in all images is **snow** in its various forms - natural snow, artificial snow, snow crystals, and snow accumulation.

**a**
  Trial 2/10: Looking at the 5 images, I can see:

1. A frost-covered tree in a snowy landscape
2. A snow-making machine spraying snow on a ski slope
3. Snow piled along a roadside
4. A detailed close-up of a snowflake crystal
5. A snow-covered mountain ridge

The common element across all images is **snow** in its various forms - natural snow, artificial snow, snowflakes, snow accumulation, and snow-covered terrain.

**a**
  Trial 3/10: Looking at the 5

{'word': 'merplin',
 'model': 'claude',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['Looking at the 5 images, I can see:\n\n1. A frost-covered tree in a snowy winter landscape\n2. A snow-making machine spraying snow on a ski slope\n3. Snow/ice buildup along a roadside\n4. A close-up macro shot of a single snowflake crystal\n5. Snow-covered mountain peaks\n\nThe common element that appears in all images is **snow** in its various forms - natural snow, artificial snow, snow crystals, and snow accumulation.\n\n**a**',
  'Looking at the 5 images, I can see:\n\n1. A frost-covered tree in a snowy landscape\n2. A snow-making machine spraying snow on a ski slope\n3. Snow piled along a roadside\n4. A detailed close-up of a snowflake crystal\n5. A snow-covered mountain ridge\n\nThe common element across all images is **snow** in its various forms - natural snow, artificial snow, snowflakes, snow accumulation, and snow-covered terrain.\n\n**a**',
  'Looking at the 5 images, I c

In [90]:
run_experiment(word_data = merplin_data, model_name = 'claude', api_key = anthropic_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with claude (elaborate prompt)...
  Trial 1/10: Looking at the images systematically:

Image 1: A frost-covered tree in a snowy field
Image 2: A snow-making machine spraying snow
Image 3: Snow/ice buildup on a road edge
Image 4: A close-up of a single snowflake crystal
Image 5: A snow-covered mountain ridge

The common observable element present in ALL images is **snow** - as a substance/material. It appears in different forms: natural snow cover, artificial snow being made, snow accumulation, crystallized snow (snowflake), and mountain snow.

- "Blizzard" (b) - Not observable; no active storm conditions shown
- "White" (c) - Too broad; includes fog, sky, ice
- "Winter" (d) - A temporal concept, not directly observable
- "Snowflakes" (e) - Only clearly visible in image 4

**a**
  Trial 2/10: I need to identify what observable feature is common across all 5 images.

Image 1: A snow-covered tree in a snowy field with

{'word': 'merplin',
 'model': 'claude',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['Looking at the images systematically:\n\nImage 1: A frost-covered tree in a snowy field\nImage 2: A snow-making machine spraying snow\nImage 3: Snow/ice buildup on a road edge\nImage 4: A close-up of a single snowflake crystal\nImage 5: A snow-covered mountain ridge\n\nThe common observable element present in ALL images is **snow** - as a substance/material. It appears in different forms: natural snow cover, artificial snow being made, snow accumulation, crystallized snow (snowflake), and mountain snow.\n\n- "Blizzard" (b) - Not observable; no active storm conditions shown\n- "White" (c) - Too broad; includes fog, sky, ice\n- "Winter" (d) - A temporal concept, not directly observable\n- "Snowflakes" (e) - Only clearly visible in image 4\n\n**a**',
  "I need to identify what observable feature is common across all 5 images.\n\nImage 1: A snow-covered tree in a snowy field with fog\n

In [91]:
run_experiment(word_data = merplin_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'minimal')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with gpt4 (minimal prompt)...
  Trial 1/10: e
  Trial 2/10: e
  Trial 3/10: e
  Trial 4/10: e
  Trial 5/10: e
  Trial 6/10: e
  Trial 7/10: e
  Trial 8/10: e
  Trial 9/10: e
  Trial 10/10: e


{'word': 'merplin',
 'model': 'gpt4',
 'prompt_type': 'minimal',
 'n_iterations': 10,
 'responses': ['e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e', 'e']}

In [92]:
run_experiment(word_data = merplin_data, model_name = 'gpt4', api_key = openai_api_key, prompt_type = 'elaborate')

Encoding 5 images for 'merplin'...

Running 10 iterations for 'merplin' with gpt4 (elaborate prompt)...
  Trial 1/10: a
  Trial 2/10: c
  Trial 3/10: c) white
  Trial 4/10: c
  Trial 5/10: a
  Trial 6/10: a
  Trial 7/10: a) snow
  Trial 8/10: c) white
  Trial 9/10: a) snow
  Trial 10/10: e


{'word': 'merplin',
 'model': 'gpt4',
 'prompt_type': 'elaborate',
 'n_iterations': 10,
 'responses': ['a',
  'c',
  'c) white',
  'c',
  'a',
  'a',
  'a) snow',
  'c) white',
  'a) snow',
  'e']}